# Day 65: Model Interpretability with SHAP
Deep Dive into Explainable AI

**Day 65 of 369-day Python & AI Learning Path** 

## Welcome to Day 65!

As machine learning models grow increasingly complex — from deep neural networks to gradient-boosted ensembles — they often become "black boxes" that make accurate predictions but offer little insight into *how* or *why* those predictions are made. This opacity poses serious challenges: how do we trust a model's decisions in healthcare, finance, or criminal justice? How do we debug a model when it fails? How do we comply with regulations like the EU AI Act or GDPR's "right to explanation"?

Model interpretability is no longer a nice-to-have; it's a critical component of responsible AI. Today, we dive deep into **SHAP (SHapley Additive exPlanations)**, a unified framework based on cooperative game theory that provides both **global** (overall model behavior) and **local** (individual prediction) explanations. SHAP has become the gold standard in explainable AI because it satisfies desirable theoretical properties — consistency, local accuracy, and missingness — that many other methods lack.

By the end of this notebook, you'll be able to generate beautiful, insightful SHAP visualizations, compare explanations across different model types, and use SHAP insights to debug models and guide feature engineering. Let's unlock the black box! 

## Table of Contents

1. [Why Model Interpretability Matters in AI/ML](#1-why-model-interpretability-matters-in-aiml)
2. [Introduction to SHAP and Shapley Values](#2-introduction-to-shap-and-shapley-values)
3. [Installing and Setting Up SHAP](#3-installing-and-setting-up-shap)
4. [Global Interpretability](#4-global-interpretability)
5. [Local Interpretability](#5-local-interpretability)
6. [Dependence Plots (Feature Interactions)](#6-dependence-plots-feature-interactions)
7. [Applying SHAP to XGBoost / LightGBM / Random Forest](#7-applying-shap-to-xgboost--lightgbm--random-forest)
8. [SHAP on Titanic Dataset (Classification)](#8-shap-on-titanic-dataset-classification)
9. [SHAP on House Prices Dataset (Regression)](#9-shap-on-house-prices-dataset-regression)
10. [Best Practices for Using SHAP in Real Projects](#10-best-practices-for-using-shap-in-real-projects)
11. [Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Key Insights](#solutions--key-insights-review-after-attempting)


## 1. Why Model Interpretability Matters in AI/ML

### The Black Box Problem
Modern ML models like XGBoost, deep neural networks, and ensemble methods can have thousands of parameters and complex non-linear interactions. While they often achieve state-of-the-art performance, their internal decision-making processes are opaque to humans.

### Key Reasons for Interpretability:
- **Trust & Transparency**: Stakeholders need to understand and trust model decisions, especially in high-stakes domains.
- **Debugging & Improvement**: Understanding *why* a model fails helps us fix data issues, feature problems, or model biases.
- **Fairness & Ethics**: Detect discriminatory patterns and ensure equitable outcomes across demographic groups.
- **Regulatory Compliance**: GDPR's "right to explanation," EU AI Act, and industry regulations increasingly require explainable AI.
- **Scientific Discovery**: Interpretable models can reveal novel insights about the underlying data-generating process.

### Types of Interpretability:
| Type | Scope | Example |
|------|-------|---------|
| **Global** | Entire model | Which features are most important overall? |
| **Local** | Single prediction | Why was *this* loan application rejected? |

SHAP provides both - making it uniquely powerful among interpretability tools.


## 2. Introduction to SHAP and Shapley Values

### What Are Shapley Values?
SHAP is based on **Shapley values** from cooperative game theory (Lloyd Shapley, 1953). Imagine a prediction is a "payout" that needs to be fairly distributed among all "players" (features). Each feature's Shapley value represents its fair contribution to the prediction, considering all possible coalitions of features.

### The SHAP Equation:
For a prediction $f(x)$, the SHAP value for feature $i$ is:

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(|N|-|S|-1)!}{|N|!} \left[ f_{S \cup \{i\}}(x_{S \cup \{i\}}) - f_S(x_S) \right]$$

Where:
- $N$ = set of all features
- $S$ = subset of features excluding $i$
- $f_S$ = model trained with subset $S$

### Key Properties:
1. **Local Accuracy**: $\sum_i \phi_i = f(x) - E[f(x)]$ (SHAP values sum to the actual prediction minus the baseline)
2. **Missingness**: If a feature doesn't change the prediction, its SHAP value is zero
3. **Consistency**: If a model changes so a feature's impact increases, its SHAP value won't decrease

### SHAP vs. Other Methods:
| Method | Type | Pros | Cons |
|--------|------|------|------|
| SHAP | Model-agnostic | Theoretically grounded, global + local | Computationally expensive |
| LIME | Model-agnostic | Fast, intuitive | Inconsistent, local only |
| Permutation | Model-agnostic | Simple, fast | No local explanations |
| Feature Importance | Built-in | Fast | Biased, no directionality |

## 3. Installing and Setting Up SHAP

Let's install the required libraries and set up our environment. We'll need `shap`, `xgboost`, `lightgbm`, and standard data science tools.

In [ ]:
# Install required libraries
!pip install shap xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn -q

# Verify installations
import shap
import xgboost as xgb
import lightgbm as lgb
import sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"SHAP version: {shap.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Initialize SHAP's JavaScript visualization (for Jupyter)
shap.initjs()


In [ ]:
# Create a synthetic dataset for initial demonstrations
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

# Classification dataset
X_cls, y_cls = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)
feature_names = [f'feature_{i}' for i in range(X_cls.shape[1])]
X_cls_df = pd.DataFrame(X_cls, columns=feature_names)

# Split data
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls_df, y_cls, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train_cls.shape}")
print(f"Test set shape: {X_test_cls.shape}")
print("\nFeature names:", feature_names)


## 4. Global Interpretability

Global interpretability helps us understand the model's overall behavior. We'll explore two key SHAP plots:

### 4.1 Summary Plot (Beeswarm)
The beeswarm plot shows the distribution of SHAP values for each feature, colored by feature value. It reveals:
- Which features are most important (vertical ordering)
- How feature values affect predictions (color: red = high, blue = low)
- The spread of impact (horizontal spread)

### 4.2 Bar Plot for Feature Importance
A straightforward ranking of features by mean absolute SHAP value.

In [ ]:
# Train a Random Forest model for global interpretability demo
from sklearn.ensemble import RandomForestClassifier

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_cls, y_train_cls)

# Create SHAP explainer
explainer_rf = shap.TreeExplainer(rf_model)

# Calculate SHAP values for test set
shap_values_rf = explainer_rf.shap_values(X_test_cls)

# Normalize SHAP output across library versions to a 2D array for class 1
if isinstance(shap_values_rf, list):
    shap_values_rf_class1 = shap_values_rf[1]
elif isinstance(shap_values_rf, np.ndarray) and shap_values_rf.ndim == 3:
    shap_values_rf_class1 = shap_values_rf[:, :, 1]
else:
    shap_values_rf_class1 = shap_values_rf

# Normalize expected value for class 1
if isinstance(explainer_rf.expected_value, (list, np.ndarray)):
    expected_value_rf_class1 = explainer_rf.expected_value[1]
else:
    expected_value_rf_class1 = explainer_rf.expected_value

print(f"SHAP values shape: {shap_values_rf_class1.shape}")
print(f"Base value (expected value): {expected_value_rf_class1}")


In [ ]:
# 4.1 Summary Plot (Beeswarm) - Global Feature Importance
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_rf_class1,
    X_test_cls,
    feature_names=feature_names,
    show=False,
    plot_size=(10, 6)
)
plt.title('SHAP Summary Plot (Beeswarm)\nFeature Impact on Model Predictions', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Interpretation:
# - Features are ranked by importance (top = most important)
# - Each dot is a single prediction's SHAP value for that feature
# - Color indicates feature value (red = high, blue = low)
# - Horizontal spread shows the range of impact
# - Example: If 'feature_0' has red dots on the right, high values of feature_0 push predictions higher


In [ ]:
# 4.2 Bar Plot for Feature Importance
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_rf_class1,
    X_test_cls,
    feature_names=feature_names,
    plot_type='bar',
    show=False,
    plot_size=(10, 6)
)
plt.title('Mean Absolute SHAP Values\nGlobal Feature Importance Ranking', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Interpretation:
# - Bars show the mean absolute SHAP value for each feature
# - Longer bars = more important features
# - This is similar to permutation importance but theoretically grounded
# - Great for reporting to non-technical stakeholders


## 5. Local Interpretability

Local interpretability explains **individual predictions** — crucial for debugging specific cases, explaining decisions to users, or auditing model behavior.

### 5.1 Force Plot
Visualizes how each feature pushes the prediction from the base value to the final output.

### 5.2 Waterfall Plot
Shows the cumulative contribution of each feature to a single prediction.

### 5.3 Decision Plot
Displays the decision path showing how features contribute cumulatively.

In [ ]:
# Select a single instance for local explanation
instance_idx = 0 # First test instance
instance = X_test_cls.iloc[instance_idx:instance_idx+1]

print(f"Instance {instance_idx} features:")
print(instance.T)
print(f"\nActual label: {y_test_cls[instance_idx]}")
print(f"Predicted probability: {rf_model.predict_proba(instance)[0][1]:.4f}")
print(f"Base value (expected probability): {expected_value_rf_class1:.4f}")


In [ ]:
# 5.1 Force Plot - Single Prediction Explanation
# Shows how features push the prediction from base value to final output
shap.force_plot(
    expected_value_rf_class1,
    shap_values_rf_class1[instance_idx],
    instance.values[0],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.title('SHAP Force Plot\nHow Features Push the Prediction', fontsize=12, pad=20)
plt.tight_layout()
plt.show()

# Interpretation:
# - Base value (left) = average prediction across all instances
# - Red arrows = features pushing prediction higher (toward positive class)
# - Blue arrows = features pushing prediction lower (toward negative class)
# - Final value (right) = model output for this instance
# - Arrow width = magnitude of impact


In [ ]:
# 5.2 Waterfall Plot - Cumulative Feature Contributions
# Best for understanding a single prediction in detail
plt.figure(figsize=(12, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_rf_class1[instance_idx],
        base_values=expected_value_rf_class1,
        data=instance.values[0],
        feature_names=feature_names
    ),
    show=False
)
plt.title('SHAP Waterfall Plot\nCumulative Contribution to Single Prediction', fontsize=12, pad=20)
plt.tight_layout()
plt.show()

# Interpretation:
# - Starts at base value (E[f(X)] = expected model output)
# - Each bar shows a feature's SHAP value (positive or negative)
# - Bars build up cumulatively to the final prediction
# - Great for detailed debugging and human-readable explanations


In [ ]:
# 5.3 Decision Plot - Decision Path Visualization
# Shows how the model decides by cumulatively adding feature contributions
plt.figure(figsize=(10, 8))
shap.decision_plot(
    expected_value_rf_class1,
    shap_values_rf_class1[instance_idx],
    instance.values[0],
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Decision Plot\nDecision Path for Single Prediction', fontsize=12, pad=20)
plt.tight_layout()
plt.show()

# Interpretation:
# - Gray line = base value
# - Colored line = cumulative sum of SHAP values
# - Each segment represents a feature contribution
# - Final point = model output
# - Useful for comparing multiple predictions side by side


## 6. Dependence Plots (Feature Interactions)

Dependence plots reveal **how a feature's value relates to its SHAP value**, and can show interactions with other features. They're essential for understanding non-linear relationships and feature interactions that the model has learned.

In [ ]:
# Dependence Plot - Feature vs. SHAP Value
# Shows relationship between feature value and its impact on predictions

# Find the most important feature from our summary plot
mean_shap = np.abs(shap_values_rf_class1).mean(axis=0)
top_feature_idx = np.argmax(mean_shap)
top_feature_name = feature_names[top_feature_idx]

print(f"Most important feature: {top_feature_name}")

# Create dependence plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: simple dependence plot
shap.dependence_plot(
    top_feature_name,
    shap_values_rf_class1,
    X_test_cls,
    ax=axes[0],
    show=False
)
axes[0].set_title(f'Dependence Plot: {top_feature_name}', fontsize=12)

# Plot 2: dependence plot with interaction
shap.dependence_plot(
    top_feature_name,
    shap_values_rf_class1,
    X_test_cls,
    interaction_index='auto',
    ax=axes[1],
    show=False
)
axes[1].set_title(f'Dependence Plot with Interaction\n{top_feature_name}', fontsize=12)

plt.suptitle('SHAP Dependence Plots: Feature Value vs. Impact', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Interpretation:
# - X-axis: feature value
# - Y-axis: SHAP value (impact on prediction)
# - Color (right plot): value of interacting feature
# - Patterns reveal linearity, non-linearity, and feature interactions


## 7. Applying SHAP to XGBoost / LightGBM / Random Forest

One of SHAP's strengths is consistency across model types. Let's train three different models on the same data and compare their SHAP explanations. This helps us:
- Validate that important features are robust across algorithms
- Identify when models disagree (potential data leakage or overfitting)
- Choose the most interpretable model for deployment

In [ ]:
# Train three different models
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Random Forest (already trained)
print("Random Forest trained")

# 2. XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X_train_cls, y_train_cls)
print("XGBoost trained")

# 3. LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train_cls, y_train_cls)
print("LightGBM trained")

# Compare performance
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model
}

print("\nModel Performance Comparison:")
print("-" * 50)
for name, model in models.items():
    y_pred = model.predict(X_test_cls)
    y_proba = model.predict_proba(X_test_cls)[:, 1]
    acc = accuracy_score(y_test_cls, y_pred)
    auc = roc_auc_score(y_test_cls, y_proba)
    print(f"{name:15s} | Accuracy: {acc:.4f} | AUC: {auc:.4f}")


In [ ]:
# Compare SHAP explanations across models
# Create explainers for each model
explainers = {
    'Random Forest': shap.TreeExplainer(rf_model),
    'XGBoost': shap.TreeExplainer(xgb_model),
    'LightGBM': shap.TreeExplainer(lgb_model)
}

# Calculate SHAP values
shap_values_dict = {}
for name, explainer in explainers.items():
    sv = explainer.shap_values(X_test_cls)
    # Handle different output formats across SHAP versions
    if isinstance(sv, list):
        shap_values_dict[name] = sv[1]
    elif isinstance(sv, np.ndarray) and sv.ndim == 3:
        shap_values_dict[name] = sv[:, :, 1]
    else:
        shap_values_dict[name] = sv

# Create comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, sv) in enumerate(shap_values_dict.items()):
    mean_abs_shap = np.abs(sv).mean(axis=0)
    sorted_idx = np.argsort(mean_abs_shap)[::-1]
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_values = mean_abs_shap[sorted_idx]

    axes[idx].barh(range(len(sorted_features)), sorted_values[::-1], color=f'C{idx}')
    axes[idx].set_yticks(range(len(sorted_features)))
    axes[idx].set_yticklabels(sorted_features[::-1])
    axes[idx].set_xlabel('Mean |SHAP Value|')
    axes[idx].set_title(f'{name}\nFeature Importance', fontsize=11)
    axes[idx].grid(axis='x', alpha=0.3)

plt.suptitle('Cross-Model SHAP Comparison\nFeature Importance Consistency', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Interpretation:
# - Consistent top features across models suggest robust signal
# - Disagreements may indicate overfitting, different biases, or interactions
# - Use this to validate feature engineering and model selection


## 8. SHAP on Titanic Dataset (Classification)

Let's apply SHAP to a real-world dataset — the classic Titanic survival prediction. This demonstrates SHAP on categorical features, missing values, and a problem with clear business interpretability requirements.

In [ ]:
# Load and prepare Titanic dataset
from sklearn.preprocessing import LabelEncoder

# Load data (using seaborn's built-in dataset)
titanic = sns.load_dataset('titanic')

# Select relevant features and handle missing values
titanic_df = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
titanic_df['age'] = titanic_df['age'].fillna(titanic_df['age'].median())
titanic_df['embarked'] = titanic_df['embarked'].fillna(titanic_df['embarked'].mode()[0])

# Encode categorical variables
le_sex = LabelEncoder()
le_embarked = LabelEncoder()
titanic_df['sex_encoded'] = le_sex.fit_transform(titanic_df['sex'])
titanic_df['embarked_encoded'] = le_embarked.fit_transform(titanic_df['embarked'])

# Prepare features and target
X_titanic = titanic_df[['pclass', 'sex_encoded', 'age', 'sibsp', 'parch', 'fare', 'embarked_encoded']]
y_titanic = titanic_df['survived']

titanic_feature_names = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X_titanic.columns = titanic_feature_names

# Split data
X_train_titanic, X_test_titanic, y_train_titanic, y_test_titanic = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42
)

print(f"Titanic dataset shape: {X_titanic.shape}")
print(f"Survival rate: {y_titanic.mean():.2%}")
print("\nFirst 5 rows:")
print(X_titanic.head())


In [ ]:
# Train XGBoost on Titanic
titanic_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
titanic_model.fit(X_train_titanic, y_train_titanic)

# Evaluate
titanic_pred = titanic_model.predict(X_test_titanic)
titanic_acc = accuracy_score(y_test_titanic, titanic_pred)
print(f"Titanic Model Accuracy: {titanic_acc:.4f}")

# SHAP analysis
titanic_explainer = shap.TreeExplainer(titanic_model)
titanic_shap_values = titanic_explainer.shap_values(X_test_titanic)

if isinstance(titanic_shap_values, list):
    titanic_shap_values = titanic_shap_values[1]
elif isinstance(titanic_shap_values, np.ndarray) and titanic_shap_values.ndim == 3:
    titanic_shap_values = titanic_shap_values[:, :, 1]

if isinstance(titanic_explainer.expected_value, (list, np.ndarray)):
    expected_value_titanic_class1 = titanic_explainer.expected_value[1]
else:
    expected_value_titanic_class1 = titanic_explainer.expected_value

print(f"SHAP values shape: {titanic_shap_values.shape}")
print(f"Base value: {expected_value_titanic_class1}")


In [ ]:
# Titanic global interpretability
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Beeswarm plot
plt.sca(axes[0])
shap.summary_plot(
    titanic_shap_values,
    X_test_titanic,
    show=False,
    plot_size=(8, 6)
)
axes[0].set_title('Titanic: SHAP Summary Plot', fontsize=12)

# Bar plot
plt.sca(axes[1])
shap.summary_plot(
    titanic_shap_values,
    X_test_titanic,
    plot_type='bar',
    show=False,
    plot_size=(8, 6)
)
axes[1].set_title('Titanic: Feature Importance (Bar)', fontsize=12)

plt.suptitle('Titanic Survival Prediction - Global Interpretability', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Key insights:
# - Sex is typically the most important feature (women prioritized)
# - Fare and Pclass show socio-economic impact
# - Age reveals "women and children first" pattern
# - Family size (SibSp + Parch) has nuanced effects


In [ ]:
# Local explanation: Why did this passenger survive (or not)?
# Pick a passenger who survived
survived_idx = np.where(y_test_titanic.values == 1)[0][0]
passenger = X_test_titanic.iloc[survived_idx]

print("Passenger Profile:")
print(f"Class: {passenger['Pclass']}")
print(f"Sex: {'Female' if passenger['Sex'] == 0 else 'Male'}")
print(f"Age: {passenger['Age']:.1f}")
print(f"Fare: ${passenger['Fare']:.2f}")
print(f"Siblings/Spouses: {passenger['SibSp']}")
print(f"Parents/Children: {passenger['Parch']}")

# Waterfall plot for this passenger
plt.figure(figsize=(12, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=titanic_shap_values[survived_idx],
        base_values=expected_value_titanic_class1,
        data=passenger.values,
        feature_names=titanic_feature_names
    ),
    show=False
)
plt.title('Why Did This Passenger Survive?\nSHAP Waterfall Plot', fontsize=12, pad=20)
plt.tight_layout()
plt.show()

# Story from the explanation:
# - Being female strongly pushes prediction toward survival
# - Higher fare adds positive contribution
# - Third class slightly reduces survival odds


## 9. SHAP on House Prices Dataset (Regression)

Now let's tackle a regression problem using the California Housing dataset. SHAP works equally well for regression — the interpretation is even more intuitive since SHAP values directly explain the predicted price in dollars.

In [ ]:
# Load California Housing dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X_housing = housing.data
y_housing = housing.target

print("California Housing Dataset:")
print(f"Shape: {X_housing.shape}")
print(f"Target: Median House Value (in $100,000s)")
print(f"Features: {list(X_housing.columns)}")
print(f"\nFirst 5 rows:")
print(X_housing.head())

# Split data
X_train_housing, X_test_housing, y_train_housing, y_test_housing = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

print(f"\nTrain set: {X_train_housing.shape}")
print(f"Test set: {X_test_housing.shape}")


In [ ]:
# Train a Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

housing_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
housing_model.fit(X_train_housing, y_train_housing)

# Evaluate
housing_pred = housing_model.predict(X_test_housing)
housing_rmse = np.sqrt(mean_squared_error(y_test_housing, housing_pred))
housing_r2 = r2_score(y_test_housing, housing_pred)

print("House Price Model Performance:")
print(f"RMSE: {housing_rmse:.4f} ($100k units)")
print(f"R^2 Score: {housing_r2:.4f}")

# SHAP analysis
housing_explainer = shap.TreeExplainer(housing_model)
housing_shap_values = housing_explainer.shap_values(X_test_housing)

print(f"\nSHAP values shape: {housing_shap_values.shape}")



In [ ]:
# Housing global interpretability
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Summary plot
plt.sca(axes[0])
shap.summary_plot(
    housing_shap_values,
    X_test_housing,
    show=False,
    plot_size=(8, 6)
)
axes[0].set_title('Housing: SHAP Summary Plot', fontsize=12)

# Bar plot
plt.sca(axes[1])
shap.summary_plot(
    housing_shap_values,
    X_test_housing,
    plot_type='bar',
    show=False,
    plot_size=(8, 6)
)
axes[1].set_title('Housing: Feature Importance', fontsize=12)

plt.suptitle('California House Price Prediction - Global Interpretability', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Key insights:
# - MedInc (median income) is typically the strongest predictor
# - Latitude/Longitude capture location premium
# - AveRooms and AveOccup show diminishing returns
# - HouseAge has non-linear effects


In [ ]:
# Local explanation: Why is this house priced at this level?
house_idx = 0
house = X_test_housing.iloc[house_idx]

print(f"House {house_idx} features:")
for feat, val in house.items():
    print(f"  {feat}: {val:.3f}")
print(f"\nActual Price: ${y_test_housing.iloc[house_idx]:.3f} (x $100k)")
print(f"Predicted Price: ${housing_pred[house_idx]:.3f} (x $100k)")

# Waterfall plot
plt.figure(figsize=(12, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=housing_shap_values[house_idx],
        base_values=housing_explainer.expected_value,
        data=house.values,
        feature_names=list(X_housing.columns)
    ),
    show=False
)
plt.title('House Price Breakdown\nWhy is this house valued at this price?', fontsize=12, pad=20)
plt.tight_layout()
plt.show()

# Business interpretation:
# - High area median income may push price up
# - Coastal location may add premium
# - Older house age may reduce value


In [ ]:
# Dependence plot for Housing - Income vs. Price Impact
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# MedInc dependence plot
shap.dependence_plot(
    'MedInc',
    housing_shap_values,
    X_test_housing,
    ax=axes[0],
    show=False
)
axes[0].set_title('MedInc vs. SHAP Value\n(Income Impact on Price)', fontsize=11)

# MedInc with HouseAge interaction
shap.dependence_plot(
    'MedInc',
    housing_shap_values,
    X_test_housing,
    interaction_index='HouseAge',
    ax=axes[1],
    show=False
)
axes[1].set_title('MedInc vs. SHAP Value\n(Color = House Age)', fontsize=11)

plt.suptitle('Housing Feature Interactions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Insights:
# - Left: clear positive relationship between income and price impact
# - Right: interaction suggests newer homes get larger premiums in high-income areas
# - Saturation appears at very high income values


## 10. Best Practices for Using SHAP in Real Projects

### Do's:
1. **Choose the Right Explainer**:
- `TreeExplainer` for tree-based models (fastest, most accurate)
- `KernelExplainer` for any model (model-agnostic, slower)
- `DeepExplainer` for neural networks
- `LinearExplainer` for linear models

2. **Use Background Data Wisely**:
- SHAP needs a background dataset to compute baseline expectations
- Use a representative sample (100-1000 instances)
- For large datasets, use `shap.sample()` or `shap.kmeans()`

3. **Handle Categorical Features**:
- Encode before SHAP, but interpret in original categories
- Use `shap.dependence_plot()` with original category names

4. **Validate with Multiple Models**:
- Consistent feature importance across algorithms = robust signal
- Disagreements warrant investigation

5. **Communicate Uncertainty**:
- Show distribution of SHAP values, not just means
- Use beeswarm plots instead of bar plots when possible

### Don'ts:
1. **Don't Trust SHAP on Correlated Features Alone**:
- Correlated features share importance
- Use `shap.utils.hclust()` to detect and group correlated features

2. **Don't Ignore the Base Value**:
- SHAP values are relative to the expected value
- Always report the baseline prediction

3. **Don't Use Too Few Background Samples**:
- Leads to noisy SHAP estimates
- Balance computation time with accuracy

4. **Don't Over-Interpret Individual SHAP Values**:
- Local explanations can be noisy
- Look for consistent patterns across multiple instances

### Production Tips:
- **Pre-compute SHAP values** for dashboard displays
- **Store feature mappings** to convert encoded values back to human-readable labels
- **Monitor SHAP distributions** over time for model drift detection
- **Use SHAP for feature selection**: Remove features with consistently near-zero SHAP values

## Hands-On Exercises

Now it's your turn! Complete these exercises to solidify your SHAP skills. Each exercise builds on the concepts covered above.

---

### Exercise 1: Generate SHAP Summary and Bar Plots
Using the synthetic classification dataset from Section 3, generate both a beeswarm summary plot and a bar plot for feature importance. Experiment with different `max_display` values to show top N features.

**Hint**: Use `shap.summary_plot()` with `plot_type='bar'`.

In [ ]:
# Exercise 1: Your code here



### Exercise 2: Create Force Plot for Individual Prediction
Select any instance from the test set and create a force plot showing how each feature pushes the prediction away from the base value. Try with both a high-confidence positive prediction and a high-confidence negative prediction.

**Hint**: Find instances where predicted probability is >0.8 and <0.2.

In [ ]:
# Exercise 2: Your code here



### Exercise 3: Waterfall Plot for Regression
Using the California Housing dataset, create a waterfall plot for the house with the highest predicted price. Identify which three features contribute most to its high valuation.

**Hint**: Find `np.argmax(housing_pred)` to get the most expensive predicted house.

In [ ]:
# Exercise 3: Your code here



### Exercise 4: Decision Plot Comparison
Create a decision plot comparing two instances from the Titanic dataset: one that survived and one that didn't. This helps visualize how different features lead to different outcomes.

**Hint**: Use `shap.decision_plot()` with two rows of SHAP values.

In [ ]:
# Exercise 4: Your code here



### Exercise 5: Analyze Feature Interactions with Dependence Plots
For the California Housing dataset, create dependence plots for `AveRooms` and `AveOccup`. Identify any non-linear relationships or interactions with other features.

**Hint**: Use `interaction_index='auto'` to automatically detect the strongest interaction.

In [ ]:
# Exercise 5: Your code here



### Exercise 6: Compare SHAP Across Three Models
Train a Logistic Regression, Random Forest, and XGBoost model on the Titanic dataset. Compare their SHAP feature importance rankings. Which features are consistently important? Which model disagrees most?

**Hint**: For Logistic Regression, use `shap.LinearExplainer` or `shap.KernelExplainer`.

In [ ]:
# Exercise 6: Your code here



### Exercise 7: SHAP-Based Feature Selection
Use SHAP values to perform feature selection on the synthetic dataset. Remove features with mean absolute SHAP value below a threshold, retrain the model, and compare performance. How many features can you remove before performance drops significantly?

**Hint**: Calculate `np.abs(shap_values).mean(axis=0)` and set a threshold.

In [ ]:
# Exercise 7: Your code here



### Exercise 8: SHAP for Model Debugging
Find an instance in the Titanic dataset where the model makes a wrong prediction. Use SHAP to explain *why* the model was wrong. Does the explanation reveal a data quality issue or a model limitation?

**Hint**: Look for false positives (predicted survived, actually died) or false negatives.

In [ ]:
# Exercise 8: Your code here



### Exercise 9: Build a Reusable SHAP Analysis Function
Create a function `analyze_model_with_shap(model, X_train, X_test, feature_names, model_type='tree')` that:
1. Creates the appropriate SHAP explainer
2. Generates summary, bar, and dependence plots
3. Returns SHAP values and feature importance rankings
4. Handles both classification and regression

**Test your function on both the Titanic and Housing datasets.**

In [ ]:
# Exercise 9: Your code here



### Exercise 10: SHAP Insights for Feature Engineering
Using SHAP dependence plots from the California Housing dataset, suggest two new engineered features that might improve model performance. For example:
- Interaction between `MedInc` and `HouseAge`
- Ratio of `AveRooms` to `AveOccup`

Create these features, retrain the model, and compare SHAP explanations before and after. Did the new features capture meaningful signal?

**Hint**: Use domain knowledge + SHAP insights to guide feature creation.

In [ ]:
# Exercise 10: Your code here



## Solutions & Key Insights (Review After Attempting)

Below are detailed solutions and interpretations for each exercise. Review these after attempting the exercises yourself.

---

### Exercise 1 Solution: Summary and Bar Plots

```python
# Summary plot with top 8 features
shap.summary_plot(shap_values_rf_class1, X_test_cls, max_display=8, show=False)
plt.title('Top 8 Features - Beeswarm')
plt.show()

# Bar plot
shap.summary_plot(shap_values_rf_class1, X_test_cls, plot_type='bar', max_display=8, show=False)
plt.title('Top 8 Features - Bar Chart')
plt.show()
```

**Key Insight**: The `max_display` parameter controls how many features to show. Beeswarm plots reveal feature value interactions (via color), while bar plots provide a clean ranking for stakeholders.

---

### Exercise 2 Solution: Force Plot

```python
# Find high-confidence predictions
proba = rf_model.predict_proba(X_test_cls)[:, 1]
high_conf_idx = np.where(proba > 0.8)[0][0]
low_conf_idx = np.where(proba < 0.2)[0][0]

# Force plot for high-confidence positive
shap.force_plot(explainer_rf.expected_value[1], 
 shap_values_rf_class1[high_conf_idx], 
 X_test_cls.iloc[high_conf_idx],
 feature_names=feature_names, matplotlib=True)
```

**Key Insight**: High-confidence predictions typically have strong pushes in one direction. Low-confidence predictions often have conflicting features (some pushing up, some pushing down).

---

### Exercise 3 Solution: Waterfall for Most Expensive House

```python
most_expensive_idx = np.argmax(housing_pred)
shap.waterfall_plot(shap.Explanation(
 values=housing_shap_values[most_expensive_idx],
 base_values=housing_explainer.expected_value,
 data=X_test_housing.iloc[most_expensive_idx].values,
 feature_names=list(X_housing.columns)
), show=False)
```

**Key Insight**: The most expensive house likely has high median income, coastal location (low latitude), and low housing density. The waterfall plot quantifies each factor's dollar contribution.

---

### Exercise 4 Solution: Decision Plot Comparison

```python
survived_idx = np.where(y_test_titanic.values == 1)[0][0]
died_idx = np.where(y_test_titanic.values == 0)[0][0]

shap.decision_plot(titanic_explainer.expected_value[1],
 titanic_shap_values[[survived_idx, died_idx]],
 X_test_titanic.iloc[[survived_idx, died_idx]],
 feature_names=titanic_feature_names,
 highlight=0) # Highlight first instance
```

**Key Insight**: Decision plots excel at comparing instances. You'll see the survivor's line trend upward (positive SHAP values dominate) while the non-survivor's line trends downward.

---

### Exercise 5 Solution: Dependence Plots

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
shap.dependence_plot('AveRooms', housing_shap_values, X_test_housing, ax=axes[0], show=False)
shap.dependence_plot('AveOccup', housing_shap_values, X_test_housing, ax=axes[1], show=False)
```

**Key Insight**: `AveRooms` often shows a saturation effect — more rooms help up to a point, then diminish. `AveOccup` (average occupancy) typically has a negative relationship with price.

---

### Exercise 6 Solution: Cross-Model Comparison

```python
from sklearn.linear_model import LogisticRegression

# Train logistic regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_titanic, y_train_titanic)

# LinearExplainer for logistic regression
lr_explainer = shap.LinearExplainer(lr_model, X_train_titanic)
lr_shap = lr_explainer.shap_values(X_test_titanic)

# Compare top 3 features across all three models
for name, sv in [('RF', titanic_shap_values), ('LR', lr_shap)]:
 importance = np.abs(sv).mean(axis=0)
 top3 = np.argsort(importance)[-3:][::-1]
 print(f"{name} top 3: {[titanic_feature_names[i] for i in top3]}")
```

**Key Insight**: Tree-based models often rank non-linear features higher, while linear models favor features with strong linear correlations. Disagreements highlight where model assumptions differ.

---

### Exercise 7 Solution: SHAP-Based Feature Selection

```python
# Calculate mean absolute SHAP values
mean_shap = np.abs(shap_values_rf_class1).mean(axis=0)
threshold = np.percentile(mean_shap, 50) # Remove bottom 50%

# Select important features
important_features = [f for f, s in zip(feature_names, mean_shap) if s > threshold]
X_train_selected = X_train_cls[important_features]
X_test_selected = X_test_cls[important_features]

# Retrain and compare
rf_selected = RandomForestClassifier(random_state=42)
rf_selected.fit(X_train_selected, y_train_cls)
print(f"Original AUC: {roc_auc_score(y_test_cls, rf_model.predict_proba(X_test_cls)[:,1]):.4f}")
print(f"Selected AUC: {roc_auc_score(y_test_cls, rf_selected.predict_proba(X_test_selected)[:,1]):.4f}")
```

**Key Insight**: SHAP-based selection often preserves performance while reducing model complexity. The threshold can be tuned based on your speed/performance trade-off needs.

---

### Exercise 8 Solution: Debugging Wrong Predictions

```python
# Find false negatives
titanic_pred = titanic_model.predict(X_test_titanic)
false_negatives = np.where((titanic_pred == 0) & (y_test_titanic.values == 1))[0]

if len(false_negatives) > 0:
 fn_idx = false_negatives[0]
 shap.waterfall_plot(shap.Explanation(
 values=titanic_shap_values[fn_idx],
 base_values=titanic_explainer.expected_value[1],
 data=X_test_titanic.iloc[fn_idx].values,
 feature_names=titanic_feature_names
 ), show=False)
 plt.title(f'False Negative: Why did the model miss this survivor?')
 plt.show()
```

**Key Insight**: Common reasons for Titanic false negatives include: male passengers (model overweights sex), 3rd class with high fare (unusual combination), or very young males (model misses "children first" nuance).

---

### Exercise 9 Solution: Reusable SHAP Function

```python
def analyze_model_with_shap(model, X_train, X_test, feature_names, model_type='tree'):
 """
 Comprehensive SHAP analysis for any model.
 
 Parameters:
-----------
 model : fitted model object
 X_train : Training features (for background data)
 X_test : Test features to explain
 feature_names : List of feature names
 model_type : 'tree', 'linear', or 'kernel'
 
 Returns:
--------
 dict : SHAP values, explainer, and feature importance
 """
# Create explainer
 if model_type == 'tree':
 explainer = shap.TreeExplainer(model)
 elif model_type == 'linear':
 explainer = shap.LinearExplainer(model, X_train)
 else:
 explainer = shap.KernelExplainer(model.predict, shap.sample(X_train, 100))
 
# Calculate SHAP values
 shap_values = explainer.shap_values(X_test)
 if isinstance(shap_values, list):
 shap_values = shap_values[1]
 
# Generate plots
 plt.figure(figsize=(10, 6))
 shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
 plt.title('SHAP Summary Plot')
 plt.show()
 
# Feature importance
 importance = pd.DataFrame({
 'feature': feature_names,
 'mean_abs_shap': np.abs(shap_values).mean(axis=0)
 }).sort_values('mean_abs_shap', ascending=False)
 
 return {
 'shap_values': shap_values,
 'explainer': explainer,
 'feature_importance': importance
 }

# Test on Titanic
results = analyze_model_with_shap(titanic_model, X_train_titanic, X_test_titanic, 
 titanic_feature_names, model_type='tree')
```

**Key Insight**: A reusable function standardizes SHAP analysis across projects, ensuring consistent methodology and easier comparison. The `model_type` parameter makes it adaptable to any algorithm.

---

### Exercise 10 Solution: Feature Engineering with SHAP

```python
# Create engineered features based on SHAP insights
X_housing_eng = X_housing.copy()

# Interaction: Income * HouseAge (older houses in rich areas = premium?)
X_housing_eng['MedInc_x_HouseAge'] = X_housing['MedInc'] * X_housing['HouseAge']

# Rooms per person (density metric)
X_housing_eng['Rooms_per_Person'] = X_housing['AveRooms'] / (X_housing['AveOccup'] + 1)

# Retrain and compare SHAP
X_train_eng, X_test_eng, _, _ = train_test_split(X_housing_eng, y_housing, test_size=0.2, random_state=42)

model_eng = GradientBoostingRegressor(random_state=42)
model_eng.fit(X_train_eng, y_train_housing)

explainer_eng = shap.TreeExplainer(model_eng)
shap_eng = explainer_eng.shap_values(X_test_eng)

# Check if new features appear in top SHAP values
importance_eng = np.abs(shap_eng).mean(axis=0)
top_features_eng = np.argsort(importance_eng)[-5:][::-1]
print("Top 5 features after engineering:")
print([X_housing_eng.columns[i] for i in top_features_eng])
```

**Key Insight**: SHAP-guided feature engineering is powerful because it reveals *what the model wishes it had*. If two features interact strongly in dependence plots, creating an explicit interaction term often improves performance and simplifies the model's task.

## Day 65 Wrap-Up

You now have strong skills in model interpretability - essential for building trustworthy AI systems. Today you mastered:

- **SHAP fundamentals** - Shapley values, local accuracy, and theoretical guarantees
- **Global explanations** - Summary plots and bar charts for overall model understanding
- **Local explanations** - Force plots, waterfall plots, and decision plots for individual predictions
- **Feature interactions** - Dependence plots that reveal non-linear relationships
- **Cross-model comparison** - Validating feature importance across XGBoost, LightGBM, and Random Forest
- **Real-world applications** - Titanic (classification) and California Housing (regression)
- **Production best practices** - Choosing explainers, handling categorical features, and monitoring drift

Remember: **An interpretable model with slightly lower accuracy can be more valuable than a black-box model.** SHAP helps balance both.

---

### Tomorrow: Day 66 - MLOps Basics and Model Deployment

We will transition from understanding models to deploying them in production. You will learn:
- Model versioning and experiment tracking with MLflow
- Packaging models for deployment (Pickle, ONNX, BentoML)
- Building REST APIs for model serving
- Monitoring model performance in production

**Next step: take your models from notebooks to production systems.**

---

*Python & AI Learning Path | Day 65 / 369*

*Keep building. Keep explaining. Keep learning.*
